In [1]:
# Interactive FMR Toy Model — X(Ψ) and 12+log10(O/H) = 8.69 + log10 X

import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import FloatSlider, VBox, HBox, HTML, interactive_output
from IPython.display import display, clear_output

# ----- Model definitions -----
def mu(psi, eps, Mstar, w, a):
    # ψ can be scalar or array
    return psi / (eps*(Mstar*(1 + w - a) + psi/eps))

def alpha(R, a, w):
    return a / (R - 1 + a - w)

def Xfun(psi, y, R, a, w, Mstar, eps):
    m = mu(psi, eps, Mstar, w, a)
    al = alpha(R, a, w)
    with np.errstate(divide='ignore', invalid='ignore'):
        out = y*(1.0 - R)/a * (1.0 - np.power(m, -al))
    # mask non-physical or undefined regions
    mask = (~np.isfinite(out)) | (a == 0) | (~np.isfinite(al))
    return np.where(mask, np.nan, out)

# ----- Sliders -----
y_s   = FloatSlider(value=0.087, min=0.01, max=0.2, step=0.001, description='y', readout_format='.3f')
R_s   = FloatSlider(value=0.79,  min=0.4,  max=0.8, step=0.005, description='R', readout_format='.3f')
a_s   = FloatSlider(value=0.3,   min=0.0,  max=3.0, step=0.01,  description='a', readout_format='.2f')
w_s   = FloatSlider(value=1.5,   min=0.0,  max=3.0, step=0.01,  description='w', readout_format='.2f')

logM_s  = FloatSlider(value=10.0, min=9.0,  max=11.0, step=0.01, description='log10 M*', readout_format='.2f')
logE_s  = FloatSlider(value=-9.0, min=-10.0,max=-8.0, step=0.01, description='log10 ε', readout_format='.2f')
logPsi0 = FloatSlider(value=0.0,  min=-2.0, max=2.0,  step=0.01, description='log10 ψ₀', readout_format='.2f')

readout = HTML()
warn    = HTML()

# ----- Update callback -----
def update(y, R, a, w, logM, logE, logPsi0):
    clear_output(wait=True)

    Mstar = 10.0**logM
    eps   = 10.0**logE
    psi0  = 10.0**logPsi0

    psi = np.logspace(-2, 2, 600)

    denom = R - 1.0 + a - w
    al = alpha(R, a, w)
    X  = Xfun(psi, y, R, a, w, Mstar, eps)

    mu_probe   = mu(psi0, eps, Mstar, w, a)
    X_probe    = Xfun(psi0, y, R, a, w, Mstar, eps)
    OH12_probe = 8.69 + np.log10(X_probe) if (np.isfinite(X_probe) and X_probe > 0) else np.nan

    # Warnings
    wtxt = []
    if np.isclose(a, 0.0):
        wtxt.append("Warning: a = 0 makes X undefined.")
    if np.isclose(denom, 0.0, atol=1e-12):
        wtxt.append("Warning: R - 1 + a - w ≈ 0 → α undefined.")
    warn.value = "<br>".join(f"<span style='color:#b22222'>{t}</span>" for t in wtxt)

    # Readout
    def sf(x, digits=5):
        return f"{x:.{digits}g}" if np.isfinite(x) else "undefined"
    readout.value = (
        f"<b>M*</b> = {sf(Mstar,3)} &nbsp; | &nbsp; "
        f"<b>ε</b> = {sf(eps,3)} &nbsp; | &nbsp; "
        f"<b>α</b> = {sf(al,4)}<br>"
        f"<b>Probe at ψ₀</b> = {sf(psi0,4)} &nbsp; | &nbsp; "
        f"<b>μ(ψ₀)</b> = {sf(mu_probe,5)} &nbsp; | &nbsp; "
        f"<b>X(ψ₀)</b> = {sf(X_probe,5)} &nbsp; | &nbsp; "
        f"<b>12+log10(O/H)</b> = {sf(OH12_probe,4)}"
    )

    # ===== UPDATED: single figure with 1 row, 2 columns =====
    fig, axs = plt.subplots(1, 2, figsize=(9, 3.6), dpi=120, constrained_layout=True)

    # Left: X(ψ) vs ψ (log x)
    ax = axs[0]
    ax.set_xscale('log')
    ax.plot(psi, X)
    if np.isfinite(X_probe):
        ax.scatter([psi0], [X_probe])
    ax.set_xlabel(r'$\Psi$')
    ax.set_ylabel(r'$X(\Psi)$')
    ax.grid(True)

    # Right: 8.69 + log10(X) vs log10(ψ)
    ax = axs[1]
    logpsi = np.log10(psi)
    with np.errstate(divide='ignore', invalid='ignore'):
        Y = 8.69 + np.log10(X)
    ax.plot(logpsi, Y)
    if np.isfinite(OH12_probe):
        ax.scatter([logM], [OH12_probe])  # marker at (log10 ψ0, OH12)
        ax.scatter([logPsi0], [OH12_probe])
    ax.set_xlabel(r'$\log_{10}\Psi$')
    ax.set_ylabel(r'$12 + \log_{10}(\mathrm{O/H})$')
    ax.set_xlim(-2.2, 2.2)
    ax.grid(True)

    plt.show()
    # =====================================

    display(readout)
    if wtxt:
        display(warn)

# ----- Wire up widgets -----
controls = VBox([
    HBox([y_s, R_s]),
    HBox([a_s, w_s]),
    HBox([logM_s, logE_s, logPsi0])
])
out = interactive_output(update, {
    'y': y_s, 'R': R_s, 'a': a_s, 'w': w_s,
    'logM': logM_s, 'logE': logE_s, 'logPsi0': logPsi0
})

display(controls, out)


Output()